# Notebook 12 — Gap-Driven Corpus Discovery

Uses **all** high-salience concepts (covered and uncovered) as seeds to discover
films and novels via Wikipedia categories. Ranks candidates by two complementary scores:

- **`total_concept_score`** — sum of composite_score for *all* concepts the work covers
  (measures overall concept density, regardless of whether already in corpus)
- **`gap_fill_score`** — sum of composite_score for *uncovered* concepts only
  (measures how much new ground the work adds)

**Algorithm:**
1. Seeds = all composite concepts with score ≥ 75, multi-word, meaningful entity type
2. For each seed: resolve canonical Wikipedia title → category prefix queries →
   find Wikipedia media categories → fetch members
3. Validate candidates as film/novel via `wikibase-shortdesc` (batch)
4. Deduplicate against existing corpus
5. Score by both total and gap metrics
6. Export `candidate_works.csv`

**All API results cached** in `data/cache/nb12/` — re-runs complete in seconds.

In [ ]:
import pandas as pd
import numpy as np
import re
import json
import time
import unicodedata
import requests
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

DATA_DIR  = Path('../data')
CACHE_DIR = DATA_DIR / 'cache' / 'nb12'
for sub in ('search', 'cats', 'members'):
    (CACHE_DIR / sub).mkdir(parents=True, exist_ok=True)

WIKI_API  = 'https://en.wikipedia.org/w/api.php'
HEADERS   = {'User-Agent': 'CulturalLiteracyProject/1.0 (research)'}
SLEEP_SEC = 0.15

# ── Seed filtering ─────────────────────────────────────────────────────────
SCORE_THRESH = 75       # composite_score >= this (for ALL concepts, not just gaps)
SEED_TYPES   = {'PERSON', 'GPE', 'EVENT', 'NORP', 'LOC', 'WORK_OF_ART', 'ORG'}

# ── Category search caps ───────────────────────────────────────────────────
MAX_CATS_PER_PREFIX  = 2
MAX_CATS_PER_CONCEPT = 4
MAX_MEMBERS_PER_CAT  = 100
MAX_PREFIXES_PER_SEED = 3

# ── Medium detection regexes on wikibase-shortdesc ────────────────────────
FILM_RE   = re.compile(r'\bfilm\b', re.I)
NOVEL_RE  = re.compile(r'\b(novel|short story collection|story collection|novella)\b', re.I)
BOOK_RE   = re.compile(r'\bbook\b', re.I)
REJECT_RE = re.compile(
    r'\b(documentary|television|TV series|miniseries|mini-series|short film|'
    r'animated series|web series|video game|album|song|play|opera|musical|'
    r'non-fiction|nonfiction|reference book|autobiography|biography|memoir|'
    r'essay|podcast|radio|manga|comic book|graphic novel|picture book)\b', re.I
)
YEAR_RE   = re.compile(r'\b(1[89]\d\d|20[012]\d)\b')


def normalize(text):
    if not isinstance(text, str):
        return ''
    s = unicodedata.normalize('NFKD', text)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    s = s.lower().strip()
    s = re.sub(r'^(the|a|an)\s+', '', s)
    s = re.sub(r"[,;:!?'\"()\[\]]", '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


print('Configuration ready.')

## 1. Load Data — All Concepts as Seeds

In [ ]:
# Load the FULL composite dataset — seeds come from here, not just gaps
composite = pd.read_csv(
    DATA_DIR / 'cultural_salience_composite_v2.csv',
    dtype={'qid': str}, low_memory=False,
)
# Load gaps separately — used only for computing gap_fill_score
gaps   = pd.read_csv(DATA_DIR / 'concept_gaps.csv')
corpus = pd.read_csv(DATA_DIR / 'works_corpus.csv')

# Gap concept set (normalized) — for scoring
gap_norms = set(gaps['concept'].str.lower().dropna())
# Gap concept score lookup
gap_score_map = gaps.set_index(gaps['concept'].str.lower())['composite_score'].to_dict()

# Corpus dedup
corpus_norm = set(corpus['norm_title'].dropna())

print(f'Composite dataset: {len(composite):,} concepts')
print(f'Gap concepts     : {len(gaps):,}')
print(f'Existing corpus  : {len(corpus):,} works')

# ── Build seeds from ALL concepts (not just gaps) ───────────────────────
# Use display_name for seeds (more readable); concept (norm) for scoring
seeds = composite[
    (composite['composite_score'] >= SCORE_THRESH) &
    (composite['entity_type'].isin(SEED_TYPES)) &
    (composite['display_name'].notna()) &
    (composite['display_name'].str.split().str.len() >= 2) &
    (~composite['display_name'].str.match(r'^Q[0-9]+')) &
    (composite['display_name'].str.len() >= 4)
].copy().drop_duplicates('display_name').sort_values(
    'composite_score', ascending=False
).reset_index(drop=True)

# Flag whether each seed is a gap concept
seeds['is_gap'] = seeds['concept'].str.lower().isin(gap_norms)

print(f'\nSeeds: {len(seeds):,} total  (score>={SCORE_THRESH}, multi-word)')
print(f'  Gap concepts     : {seeds["is_gap"].sum():,}')
print(f'  Covered concepts : {(~seeds["is_gap"]).sum():,}')
print(f'\nBy entity type:')
print(seeds['entity_type'].value_counts().to_string())
print(f'\nTop 20 seeds:')
print(seeds[['display_name','entity_type','composite_score','is_gap']].head(20).to_string(index=False))

## 2. Wikipedia API Helpers

In [ ]:
def _wiki_get(params, retries=3):
    params.setdefault('format', 'json')
    for attempt in range(retries):
        try:
            r = requests.get(WIKI_API, params=params, headers=HEADERS, timeout=15)
            r.raise_for_status()
            return r.json()
        except Exception:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return {}


def search_wiki(query, limit=1):
    data = _wiki_get({'action': 'query', 'list': 'search',
                      'srsearch': query, 'srlimit': limit})
    try:
        return data['query']['search'][0]['title']
    except (KeyError, IndexError):
        return None


def find_media_categories(prefix, limit=5):
    data = _wiki_get({'action': 'query', 'list': 'allcategories',
                      'acprefix': prefix, 'aclimit': limit})
    cats = [c['*'] for c in data.get('query', {}).get('allcategories', [])]
    media_kw = {'film', 'novel', 'fiction', 'book', 'adapt', 'literature',
                'cinema', 'screen', 'story', 'stories'}
    return [c for c in cats if any(k in c.lower() for k in media_kw)]


def fetch_category_members(cat_title, limit=100):
    data = _wiki_get({'action': 'query', 'list': 'categorymembers',
                      'cmtitle': f'Category:{cat_title}',
                      'cmlimit': limit, 'cmnamespace': 0})
    return [m['title'] for m in data.get('query', {}).get('categorymembers', [])]


def batch_fetch_shortdesc(titles):
    result = {}
    for i in range(0, len(titles), 50):
        batch = titles[i:i+50]
        data  = _wiki_get({'action': 'query', 'titles': '|'.join(batch),
                           'prop': 'pageprops', 'ppprop': 'wikibase-shortdesc'})
        for page in data.get('query', {}).get('pages', {}).values():
            result[page.get('title', '')] = page.get('pageprops', {}).get('wikibase-shortdesc', '')
        time.sleep(SLEEP_SEC)
    return result


def classify_medium(shortdesc):
    if not shortdesc or REJECT_RE.search(shortdesc):
        return None
    if FILM_RE.search(shortdesc):
        return 'film'
    if NOVEL_RE.search(shortdesc):
        return 'novel'
    if BOOK_RE.search(shortdesc) and not FILM_RE.search(shortdesc):
        return 'novel'
    return None


print('API helpers ready.')

## 3. Cache Helpers

In [ ]:
def _safe_key(s, maxlen=80):
    return re.sub(r'[^\w\s-]', '_', str(s).lower())[:maxlen]

def _load(path):
    try:
        return json.loads(Path(path).read_text(encoding='utf-8'))
    except Exception:
        return None

def _save(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False), encoding='utf-8')

def cached_search(display_name):
    path = CACHE_DIR / 'search' / f'{_safe_key(display_name)}.json'
    cached = _load(path)
    if cached is not None:
        return cached.get('canonical')
    canonical = search_wiki(display_name)
    _save(path, {'display_name': display_name, 'canonical': canonical})
    time.sleep(SLEEP_SEC)
    return canonical

def cached_find_cats(prefix):
    path = CACHE_DIR / 'cats' / f'{_safe_key(prefix)}.json'
    cached = _load(path)
    if cached is not None:
        return cached.get('cats', [])
    cats = find_media_categories(prefix)
    _save(path, {'prefix': prefix, 'cats': cats})
    time.sleep(SLEEP_SEC)
    return cats

def cached_fetch_members(cat_name):
    path = CACHE_DIR / 'members' / f'{_safe_key(cat_name)}.json'
    cached = _load(path)
    if cached is not None:
        return cached.get('members', [])
    members = fetch_category_members(cat_name)
    _save(path, {'cat': cat_name, 'members': members})
    time.sleep(SLEEP_SEC)
    return members

# Report existing cache (carries over from any previous runs)
for sub in ('search', 'cats', 'members'):
    n = len(list((CACHE_DIR / sub).glob('*.json')))
    print(f'Cache/{sub}: {n:,} files')

## 4. Category Prefix Generation by Entity Type

In [ ]:
def get_prefixes(canonical, entity_type):
    c = canonical
    if entity_type == 'PERSON':
        return [f'Films about {c}', f'Books about {c}', f'{c} in fiction']
    elif entity_type == 'WORK_OF_ART':
        return [f'Films based on {c}', f'Adaptations of {c}', f'Novels based on {c}']
    elif entity_type == 'EVENT':
        return [f'{c} films', f'{c} fiction', f'Films about {c}', f'{c} in fiction']
    elif entity_type == 'GPE':
        return [f'Films set in {c}', f'Novels set in {c}', f'Films set in the {c}']
    elif entity_type == 'LOC':
        return [f'Films set in {c}', f'{c} in fiction', f'Novels set in {c}']
    elif entity_type == 'NORP':
        return [f'Films about {c}', f'{c} films', f'{c}-language films']
    elif entity_type == 'ORG':
        return [f'Films about {c}', f'Novels about {c}', f'Films set in {c}']
    else:
        return [f'Films about {c}']

print('Prefix generation ready.')
print('Sample prefixes:')
for et, name in [('PERSON','Theodore Roosevelt'), ('WORK_OF_ART',"Hamlet"),
                 ('EVENT','World War II'), ('GPE','Ancient Rome')]:
    print(f'  {et:12s} "{name}" -> {get_prefixes(name, et)[:2]}')

## 5. Discovery Loop

In [ ]:
# candidate_map stores per-candidate:
#   total_score  — sum of composite_score for ALL concepts that led here
#   gap_score    — sum of composite_score for only GAP concepts that led here
#   all_concepts — every display_name that discovered this work
#   gap_concepts — only the gap concept display_names
candidate_map = defaultdict(lambda: {
    'total_score': 0.0, 'gap_score': 0.0,
    'all_concepts': [], 'gap_concepts': [],
})

_REJECT_TITLE = re.compile(
    r'^(List of|Category:|Template:|Wikipedia:|Portal:)|\(disambiguation\)$', re.I
)

t0          = time.time()
n_hits      = 0
n_api_calls = 0

for i, (_, seed) in enumerate(seeds.iterrows()):
    display     = seed['display_name']
    entity_type = seed['entity_type']
    score       = float(seed['composite_score'])
    is_gap      = bool(seed['is_gap'])

    # Step 1: resolve canonical Wikipedia title
    canonical = cached_search(display)
    n_api_calls += 1
    if not canonical:
        continue

    # Step 2: build prefix queries
    prefixes = get_prefixes(canonical, entity_type)[:MAX_PREFIXES_PER_SEED]

    # Step 3: find matching media categories
    found_cats = []
    for prefix in prefixes:
        cats = cached_find_cats(prefix)
        n_api_calls += 1
        for cat in cats[:MAX_CATS_PER_PREFIX]:
            if cat not in found_cats:
                found_cats.append(cat)
    found_cats = found_cats[:MAX_CATS_PER_CONCEPT]

    if not found_cats:
        continue
    n_hits += 1

    # Step 4: fetch members and accumulate scores
    for cat in found_cats:
        members = cached_fetch_members(cat)
        n_api_calls += 1
        for title in members:
            if _REJECT_TITLE.search(title):
                continue
            cand = candidate_map[title]
            cand['total_score']  += score
            cand['all_concepts'].append(display)
            if is_gap:
                cand['gap_score']  += score
                cand['gap_concepts'].append(display)

    if (i + 1) % 500 == 0:
        elapsed = time.time() - t0
        rate    = (i + 1) / elapsed
        eta     = (len(seeds) - i - 1) / max(rate, 0.001)
        print(f'  [{i+1:5d}/{len(seeds)}] {elapsed:5.0f}s  '
              f'rate={rate:.1f}/s  ETA~{eta:.0f}s  '
              f'hits={n_hits}  candidates={len(candidate_map):,}')

elapsed = time.time() - t0
print(f'\nDiscovery complete in {elapsed:.0f}s')
print(f'  Seeds with category results: {n_hits:,} / {len(seeds):,} ({100*n_hits/len(seeds):.1f}%)')
print(f'  Unique candidate titles    : {len(candidate_map):,}')

## 6. Validate Candidates via Shortdesc

In [ ]:
all_titles = list(candidate_map.keys())
print(f'Batch-validating {len(all_titles):,} candidate titles...')
t0       = time.time()
desc_map = batch_fetch_shortdesc(all_titles)
elapsed  = time.time() - t0
print(f'Done in {elapsed:.0f}s')

_YEAR_SUFFIX   = re.compile(r'\s*\((\d{4})\s*[^)]*\)\s*$')
_MEDIUM_SUFFIX = re.compile(r'\s*\((film|novel|book|movie)\)\s*$', re.I)

rows = []
for title, info in candidate_map.items():
    desc   = desc_map.get(title, '')
    medium = classify_medium(desc)
    if medium is None:
        continue

    year_m = _YEAR_SUFFIX.search(title)
    year   = int(year_m.group(1)) if year_m else None
    if year is None:
        ym2  = YEAR_RE.search(desc)
        year = int(ym2.group(1)) if ym2 else None

    clean = _YEAR_SUFFIX.sub('', title).strip()
    clean = _MEDIUM_SUFFIX.sub('', clean).strip()
    norm  = normalize(clean)

    if norm in corpus_norm:
        continue

    all_c = list(dict.fromkeys(info['all_concepts']))
    gap_c = list(dict.fromkeys(info['gap_concepts']))

    rows.append({
        'wiki_title'        : title,
        'clean_title'       : clean,
        'norm_title'        : norm,
        'medium'            : medium,
        'year'              : year,
        'shortdesc'         : desc,
        'total_concept_score': round(info['total_score'], 2),
        'gap_fill_score'    : round(info['gap_score'], 2),
        'n_total_concepts'  : len(all_c),
        'n_gap_concepts'    : len(gap_c),
        'top_concept'       : all_c[0] if all_c else '',
        'all_concepts'      : '|'.join(all_c),
        'gap_concepts'      : '|'.join(gap_c),
    })

candidates = pd.DataFrame(rows).sort_values(
    'total_concept_score', ascending=False
).reset_index(drop=True)

print(f'\nCandidates after validation + corpus dedup:')
print(f'  Total  : {len(candidates):,}')
print(f'  Films  : {(candidates["medium"]=="film").sum():,}')
print(f'  Novels : {(candidates["medium"]=="novel").sum():,}')
print(f'\ntotal_concept_score distribution:')
print(candidates['total_concept_score'].describe(percentiles=[.5,.75,.9,.95,.99]).to_string())
print(f'\ngap_fill_score distribution:')
print(candidates['gap_fill_score'].describe(percentiles=[.5,.75,.9,.95,.99]).to_string())

## 7. Analysis

In [ ]:
cols = ['wiki_title','year','total_concept_score','gap_fill_score','n_total_concepts','n_gap_concepts','top_concept']

print('TOP 40 FILMS by total_concept_score (concept-dense overall):')
top_films = candidates[candidates['medium']=='film'].head(40)
print(top_films[cols].to_string(index=False, float_format='{:.1f}'.format))

print('\nTOP 40 NOVELS by total_concept_score:')
top_novels = candidates[candidates['medium']=='novel'].head(40)
print(top_novels[cols].to_string(index=False, float_format='{:.1f}'.format))

In [ ]:
print('TOP 40 FILMS by gap_fill_score (most NEW coverage):')
top_gap_films = candidates[candidates['medium']=='film'].sort_values('gap_fill_score', ascending=False).head(40)
print(top_gap_films[cols].to_string(index=False, float_format='{:.1f}'.format))

print('\nFilms where total >> gap (concept-dense but mostly covers already-known concepts):')
# These are works that reinforce important concepts already in corpus — good for curriculum depth
candidates['redundancy_ratio'] = candidates['gap_fill_score'] / candidates['total_concept_score'].clip(lower=1)
deep_cover = candidates[
    (candidates['medium']=='film') &
    (candidates['total_concept_score'] >= candidates['total_concept_score'].quantile(0.75)) &
    (candidates['redundancy_ratio'] < 0.3)
].sort_values('total_concept_score', ascending=False).head(20)
print(deep_cover[['wiki_title','year','total_concept_score','gap_fill_score','redundancy_ratio']]
      .to_string(index=False, float_format='{:.2f}'.format))

In [ ]:
# Scatter: total_concept_score vs gap_fill_score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for med, col in [('film','#1f77b4'),('novel','#d62728')]:
    sub = candidates[candidates['medium']==med]
    ax.scatter(sub['total_concept_score'], sub['gap_fill_score'],
               c=col, alpha=0.3, s=15, label=med.capitalize())
ax.set_xlabel('Total Concept Score (all high-salience concepts)')
ax.set_ylabel('Gap Fill Score (uncovered concepts only)')
ax.set_title('Candidate Works: Total vs Gap Coverage')
ax.legend()

# Top 15 labels
for _, r in candidates.head(15).iterrows():
    ax.annotate(r['clean_title'][:25], (r['total_concept_score'], r['gap_fill_score']),
                fontsize=6, alpha=0.8, xytext=(3,2), textcoords='offset points')

ax = axes[1]
ax.hist(candidates[candidates['medium']=='film']['total_concept_score'],
        bins=40, alpha=0.6, color='steelblue', label='Films')
ax.hist(candidates[candidates['medium']=='novel']['total_concept_score'],
        bins=40, alpha=0.6, color='tomato', label='Novels')
ax.set_xlabel('total_concept_score')
ax.set_title('Score Distribution by Medium')
ax.legend()

plt.tight_layout()
plt.savefig(DATA_DIR / 'candidate_works_distribution.png', dpi=150)
plt.show()

# Medium summary
print('\nMedium breakdown:')
print(candidates.groupby('medium').agg(
    count=('wiki_title','count'),
    avg_total=('total_concept_score','mean'),
    avg_gap=('gap_fill_score','mean'),
    avg_n_concepts=('n_total_concepts','mean'),
).to_string(float_format='{:.1f}'.format))

## 8. Export

In [ ]:
out_path = DATA_DIR / 'candidate_works.csv'
candidates.drop(columns=['redundancy_ratio'], errors='ignore').to_csv(out_path, index=False)
print(f'Saved {len(candidates):,} candidates -> {out_path}')
print(f'  Films  : {(candidates["medium"]=="film").sum():,}')
print(f'  Novels : {(candidates["medium"]=="novel").sum():,}')

print(f'\nCache files:')
for sub in ('search', 'cats', 'members'):
    n = len(list((CACHE_DIR / sub).glob('*.json')))
    print(f'  nb12/{sub}/: {n:,} files')

print(f'\n=== SUMMARY ===')
print(f'Seeds (all concepts, score>={SCORE_THRESH}): {len(seeds):,}')
print(f'  Gap seeds     : {seeds["is_gap"].sum():,}')
print(f'  Covered seeds : {(~seeds["is_gap"]).sum():,}')
p90 = candidates['total_concept_score'].quantile(0.9)
top = candidates[candidates['total_concept_score'] >= p90]
print(f'\nTop 10% by total_concept_score (>={p90:.0f}): {len(top):,} works')
print(f'  Films  : {(top["medium"]=="film").sum():,}')
print(f'  Novels : {(top["medium"]=="novel").sum():,}')